In [0]:
%python
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils.utils_merge_into_tables import upsert_data

In [0]:
%python

catalog_olist = dbutils.widgets.get("catalog_olist")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")
table_silver = dbutils.widgets.get("table_silver")
output_table = dbutils.widgets.get("output_table")
sk_table_olist = dbutils.widgets.get('sk_table')

In [0]:
CREATE OR REPLACE TEMPORARY VIEW silver_table_customer_olist AS
SELECT
customer_unique_id,
customer_zip_code_prefix,
customer_city,
customer_state,
silver_update_date
FROM ${catalog_olist}.${schema_silver}.${table_silver};

In [0]:
CREATE OR REPLACE TEMPORARY VIEW silver_table_customer_olist_deduplicated AS
SELECT
*
FROM silver_table_customer_olist
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY silver_update_date DESC, customer_zip_code_prefix DESC) = 1;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW ${output_table} AS 
SELECT
XXHASH64(customer_unique_id) AS SK_CUSTOMER,
customer_unique_id AS ID_CUSTOMER,
CONCAT_WS(' - ', UPPER(customer_city), UPPER(customer_state)) AS CITY_STATE,
customer_zip_code_prefix AS ZIP_CODE
FROM silver_table_customer_olist_deduplicated;

## Merge Table

In [0]:
%run ../setup/00_aws_connection

In [0]:
%python
df_dim_customers = spark.table(output_table)
full_table_name =  f'{catalog_olist}.{schema_gold}.{output_table}'

upsert_data(df_dim_customers, full_table_name,sk_table_olist,name_bucket,layer='gold')
